# Processing sqlite tables 

In [21]:
import sqlite3
import json
import requests

In [22]:
# ---- Core disease / concept phrases (from your list + MCAS + RPL) ----

RAW_TERMS = [
    # MCAS / mast cell & allergy-ish
    "Mastocytosis",
    "Mast Cells",
    "Hypersensitivity",
    "Anaphylaxis",
    "Food Hypersensitivity",
    "Drug Hypersensitivity",
    "Histamine",
    "Histamine Release",

    # Autoimmune / immune
    "Autoimmune Diseases",
    "Autoimmunity",
    "Immune System Diseases",
    "Systemic Inflammatory Response Syndrome",
    "Cytokines",
    "Inflammation Mediators",
    "Interleukin-6",
    "TNF-alpha",
    "C-reactive protein",
]

# ---- Manual “extra” variants / abbreviations ----

EXTRA_VARIANTS = {
    "Systemic Inflammatory Response Syndrome": ["sirs"],
    "C-reactive protein": ["crp"],
}

In [23]:
def build_relevant_keywords(raw_terms, extra_variants):
    kws = set()

    for term in raw_terms:
        t = term.strip()
        if not t:
            continue
        base = t.lower()
        kws.add(base)

        # space <-> dash variants
        kws.add(base.replace("-", " "))
        kws.add(base.replace(" ", "-"))

        # crude slash handling (e.g. "Endometrium/immunology")
        kws.add(base.replace("/", " "))
        kws.add(base.replace("/", "-"))

    # add manual variants
    for canonical, variants in extra_variants.items():
        for v in variants:
            vlow = v.lower()
            kws.add(vlow)
            kws.add(vlow.replace("-", " "))
            kws.add(vlow.replace(" ", "-"))

    # remove empty strings if any
    kws.discard("")
    return sorted(kws)

RELEVANT_KEYWORDS = build_relevant_keywords(RAW_TERMS, EXTRA_VARIANTS)

print(f"{len(RELEVANT_KEYWORDS)} keywords")
print(RELEVANT_KEYWORDS[:40])  # peek if you want

31 keywords
['anaphylaxis', 'autoimmune diseases', 'autoimmune-diseases', 'autoimmunity', 'c reactive protein', 'c-reactive protein', 'c-reactive-protein', 'crp', 'cytokines', 'drug hypersensitivity', 'drug-hypersensitivity', 'food hypersensitivity', 'food-hypersensitivity', 'histamine', 'histamine release', 'histamine-release', 'hypersensitivity', 'immune system diseases', 'immune-system-diseases', 'inflammation mediators', 'inflammation-mediators', 'interleukin 6', 'interleukin-6', 'mast cells', 'mast-cells', 'mastocytosis', 'sirs', 'systemic inflammatory response syndrome', 'systemic-inflammatory-response-syndrome', 'tnf alpha', 'tnf-alpha']


In [31]:
def is_relevant_disease_name(disease_name: str) -> bool:
    dn = disease_name.lower()
    return any(kw in dn for kw in RELEVANT_KEYWORDS)

# Cleaning function
def remove_mesh(cur, extra):
    command = """
    UPDATE mesh2disease
    SET disease = TRIM(REPLACE(disease, '{0}', ''))
    WHERE disease LIKE '%{0}';
    """.format(extra)
    
    cur.execute(command)

In [39]:
# Set up connection and make initial tables
db_path = "pubtator_bioc0.sqlite"

conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM documents") #total documents imported
# cur.execute("""
# CREATE TABLE IF NOT EXISTS pubtatorBioc (
#     type    TEXT,
#     journal TEXT,
#     identifier TEXT,
#     authors TEXT
# )
# """)

tables = cur.fetchall()
print(tables)

[(5400,)]


In [40]:
cur.execute("SELECT COUNT(*) FROM passages")
tables = cur.fetchall()
print(tables)

[(61541,)]


In [41]:
cur.execute("SELECT COUNT(*) FROM sentences")
tables = cur.fetchall()
print(tables)
#these are subset of passages ? - figure out how to get info from passages

[(0,)]


In [42]:
cur.execute("SELECT COUNT(*) FROM annotations")
tables = cur.fetchall()
print(tables)

[(166642,)]


In [43]:
cur.execute("SELECT COUNT(*) FROM relations")
tables = cur.fetchall()
print(tables)

[(2936,)]


In [44]:
cur.execute("SELECT COUNT(*) FROM relation_nodes")
tables = cur.fetchall()
print(tables)

[(2887,)]


In [45]:
cur.execute("SELECT COUNT(*) FROM processed_members")
tables = cur.fetchall()
print(tables)

[(9,)]


In [8]:
batch_pub2mesh = []
batch_mesh2disease = []
batch_size = 5  # easily tweakable

def is_relevant_disease_name(disease_name: str) -> bool:
    #print(disease_name)
    dn = disease_name.lower()
    #dn = disease_name
    return any(kw in dn for kw in RELEVANT_KEYWORDS)